# BODAQS Simple Suspension Metrics

Special-purpose notebook focused on canonical suspension views (no event-derived metrics).

- Two-column tiled layout
- Left: front suspension
- Right: rear suspension


In [1]:
from IPython.display import display
from bodaqs_analysis.widgets.session_selector import make_session_selector

sel = make_session_selector(artifacts_dir="artifacts", select_first_by_default=True)
display(sel["ui"])

store = sel["store"]


## Displacement Histograms

In [2]:
from typing import Any, Mapping

import ipywidgets as W
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

from bodaqs_analysis.widgets.contracts import entity_snapshot_from_handle
from bodaqs_analysis.widgets.loaders import make_session_loader
from bodaqs_analysis.widgets.session_selector import attach_refresh


def _norm_candidates_for_side(signals: Mapping[str, Mapping[str, Any]], side: str) -> list[str]:
    side_l = str(side).strip().lower()
    candidates: list[tuple[tuple[int, int, int, int], str]] = []

    for col, info in signals.items():
        if not isinstance(info, Mapping):
            continue
        if str(info.get('kind') or '').strip().lower() == 'qc':
            continue

        domain = str(info.get('domain') or '').strip().lower()
        if domain != 'suspension':
            continue

        raw_ops = info.get('op_chain') or []
        if isinstance(raw_ops, (list, tuple)):
            ops = [str(x).strip().lower() for x in raw_ops if str(x).strip()]
        else:
            ops = [str(raw_ops).strip().lower()] if str(raw_ops).strip() else []
        if 'norm' not in ops:
            continue

        col_s = str(col)
        col_l = col_s.lower()
        sensor = str(info.get('sensor') or '').strip().lower()
        if side_l not in col_l and side_l not in sensor:
            continue

        quantity = str(info.get('quantity') or '').strip().lower()
        unit = str(info.get('unit') or '').strip().lower()
        score = (
            1 if quantity == 'disp_norm' else 0,
            1 if unit == '1' else 0,
            1 if sensor.startswith(f'{side_l}_') else 0,
            1 if 'zeroed' in ops else 0,
        )
        candidates.append((score, col_s))

    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return [col for _score, col in candidates]


def _mm_candidates_for_side(signals: Mapping[str, Mapping[str, Any]], side: str) -> list[str]:
    side_l = str(side).strip().lower()
    candidates: list[tuple[tuple[int, int, int, int], str]] = []

    for col, info in signals.items():
        if not isinstance(info, Mapping):
            continue
        if str(info.get('kind') or '').strip().lower() == 'qc':
            continue

        domain = str(info.get('domain') or '').strip().lower()
        if domain != 'suspension':
            continue

        col_s = str(col)
        col_l = col_s.lower()
        sensor = str(info.get('sensor') or '').strip().lower()
        if side_l not in col_l and side_l not in sensor:
            continue

        quantity = str(info.get('quantity') or '').strip().lower()
        unit = str(info.get('unit') or '').strip().lower()
        if unit != 'mm':
            continue

        raw_ops = info.get('op_chain') or []
        if isinstance(raw_ops, (list, tuple)):
            ops = [str(x).strip().lower() for x in raw_ops if str(x).strip()]
        else:
            ops = [str(raw_ops).strip().lower()] if str(raw_ops).strip() else []

        score = (
            1 if quantity == 'disp' else 0,
            1 if sensor.startswith(f'{side_l}_') else 0,
            1 if 'zeroed' in ops else 0,
            1 if 'norm' not in ops else 0,
        )
        candidates.append((score, col_s))

    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return [col for _score, col in candidates]


def _resolve_displacement_sources_by_session(*, session_loader, session_keys: list[str], side: str) -> tuple[dict[str, str], dict[str, str], list[str], list[str]]:
    norm_by_session: dict[str, str] = {}
    mm_by_session: dict[str, str] = {}
    missing_norm: list[str] = []
    missing_mm: list[str] = []

    for sk in map(str, session_keys):
        sess = session_loader(sk)
        meta = (sess or {}).get('meta') or {}
        signals = meta.get('signals') or {}
        if not isinstance(signals, Mapping):
            signals = {}

        norm_candidates = _norm_candidates_for_side(signals, side)
        mm_candidates = _mm_candidates_for_side(signals, side)

        if norm_candidates:
            norm_by_session[sk] = norm_candidates[0]
        else:
            missing_norm.append(sk)

        if mm_candidates:
            mm_by_session[sk] = mm_candidates[0]
        else:
            missing_mm.append(sk)

    return norm_by_session, mm_by_session, missing_norm, missing_mm


def _fmt_metric(v: float) -> str:
    if np.isnan(v):
        return 'nan'
    return f'{v:.3f}'


def _fmt_pct_mm(norm_v: float, mm_v: float) -> str:
    if np.isnan(norm_v) or np.isnan(mm_v):
        return 'nan'
    return f'{(100.0 * float(norm_v)):.1f}% ({float(mm_v):.1f} mm)'


def _extract_series(df: pd.DataFrame, col: str, *, include_inactive: bool = False) -> np.ndarray:
    if col not in df.columns:
        return np.array([], dtype=float)
    s = pd.to_numeric(df[col], errors='coerce')
    if (not include_inactive) and ('active_mask_qc' in df.columns):
        s = s[df['active_mask_qc'].astype(bool)]
    vals = s.to_numpy(dtype=float, copy=False)
    vals = vals[np.isfinite(vals)]
    return vals


def _extract_paired_series(
    df: pd.DataFrame,
    *,
    norm_col: str,
    mm_col: str,
    include_inactive: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    if norm_col not in df.columns or mm_col not in df.columns:
        return np.array([], dtype=float), np.array([], dtype=float)

    s_norm = pd.to_numeric(df[norm_col], errors='coerce')
    s_mm = pd.to_numeric(df[mm_col], errors='coerce')

    if (not include_inactive) and ('active_mask_qc' in df.columns):
        mask_active = df['active_mask_qc'].astype(bool)
        s_norm = s_norm[mask_active]
        s_mm = s_mm[mask_active]

    a_norm = s_norm.to_numpy(dtype=float, copy=False)
    a_mm = s_mm.to_numpy(dtype=float, copy=False)
    mask = np.isfinite(a_norm) & np.isfinite(a_mm)
    return a_norm[mask], a_mm[mask]


def _paired_disp_metrics(norm_vals: np.ndarray, mm_vals: np.ndarray, trim_cutoff: float = 0.05) -> dict[str, float | bool]:
    n = np.asarray(norm_vals, dtype=float)
    m = np.asarray(mm_vals, dtype=float)

    paired_ok = np.isfinite(n) & np.isfinite(m)
    n = n[paired_ok]
    m = m[paired_ok]

    if n.size == 0:
        return {
            'insufficient': True,
            'n': 0.0,
            'q50_n': np.nan,
            'q95_n': np.nan,
            'q100_n': np.nan,
            'iqr_n': np.nan,
            'skew_n': np.nan,
            'q50_mm': np.nan,
            'q95_mm': np.nan,
            'q100_mm': np.nan,
            'iqr_mm': np.nan,
            'skew_mm': np.nan,
        }

    keep = n >= float(trim_cutoff)
    n = n[keep]
    m = m[keep]

    if n.size == 0:
        return {
            'insufficient': True,
            'n': 0.0,
            'q50_n': np.nan,
            'q95_n': np.nan,
            'q100_n': np.nan,
            'iqr_n': np.nan,
            'skew_n': np.nan,
            'q50_mm': np.nan,
            'q95_mm': np.nan,
            'q100_mm': np.nan,
            'iqr_mm': np.nan,
            'skew_mm': np.nan,
        }

    q25_n, q50_n, q75_n, q95_n, q100_n = np.quantile(n, [0.25, 0.5, 0.75, 0.95, 1.0])
    q25_mm, q50_mm, q75_mm, q95_mm, q100_mm = np.quantile(m, [0.25, 0.5, 0.75, 0.95, 1.0])

    iqr_n = float(q75_n - q25_n)
    iqr_mm = float(q75_mm - q25_mm)

    skew_n = np.nan if iqr_n <= 0 else float((q75_n + q25_n - (2.0 * q50_n)) / iqr_n)
    skew_mm = np.nan if iqr_mm <= 0 else float((q75_mm + q25_mm - (2.0 * q50_mm)) / iqr_mm)

    return {
        'insufficient': False,
        'n': float(n.size),
        'q50_n': float(q50_n),
        'q95_n': float(q95_n),
        'q100_n': float(q100_n),
        'iqr_n': float(iqr_n),
        'skew_n': float(skew_n),
        'q50_mm': float(q50_mm),
        'q95_mm': float(q95_mm),
        'q100_mm': float(q100_mm),
        'iqr_mm': float(iqr_mm),
        'skew_mm': float(skew_mm),
    }


def make_displacement_hist_tile_rebuilder(
    *,
    sel,
    side: str,
    title: str,
    bins: int = 50,
    trim_cutoff: float = 0.05,
    y_shared: dict[str, float] | None = None,
    y_key: str | None = None,
    show_engineering_getter = None,
):
    out = W.Output(layout=W.Layout(border='1px solid #d9d9d9', padding='8px', width='100%'))
    state: dict[str, Any] = {}
    side_l = str(side).strip().lower()
    shared = y_shared if isinstance(y_shared, dict) else {}

    def rebuild() -> None:
        snapshot = entity_snapshot_from_handle(sel)
        selected_entities = list(snapshot.selected_entities)
        show_engineering = bool(show_engineering_getter()) if callable(show_engineering_getter) else False

        with out:
            clear_output(wait=True)
            display(W.HTML(f"<h4 style='margin:0 0 8px 0;'>{title}</h4>"))

            if not selected_entities:
                print('No entities selected.')
                return

            key_to_ref = dict(snapshot.key_to_ref)
            base_loader = make_session_loader(store=sel['store'], key_to_ref=key_to_ref)
            session_keys = list(map(str, snapshot.expanded_session_keys))
            norm_by_session, mm_by_session, missing_norm, missing_mm = _resolve_displacement_sources_by_session(
                session_loader=base_loader,
                session_keys=session_keys,
                side=side_l,
            )

            if show_engineering and (not mm_by_session):
                print(
                    'No matching engineering-unit suspension displacement signal found '
                    f'for side={side_l!r} in the current selection.'
                )
                return

            if (not show_engineering) and (not norm_by_session):
                print(
                    'No matching normalized suspension displacement signal found '
                    f'for side={side_l!r} in the current selection.'
                )
                return

            notes: list[str] = []
            if missing_norm:
                notes.append(f"norm missing in {len(missing_norm)} session(s)")
            if missing_mm:
                notes.append(f"mm missing in {len(missing_mm)} session(s)")
            if notes:
                display(W.HTML(f"<small><b>Note:</b> {'; '.join(notes)}.</small>"))

            hist_values_by_entity: dict[str, np.ndarray] = {}
            paired_norm_by_entity: dict[str, np.ndarray] = {}
            paired_mm_by_entity: dict[str, np.ndarray] = {}

            for entity in selected_entities:
                entity_key = str(entity.entity_key)
                label = str(entity.label)
                members = snapshot.entity_to_effective_members.get(entity_key, [entity_key])

                hist_chunks: list[np.ndarray] = []
                norm_chunks: list[np.ndarray] = []
                mm_chunks: list[np.ndarray] = []

                for sk in map(str, members):
                    sess = base_loader(sk)
                    df = (sess or {}).get('df')
                    if not isinstance(df, pd.DataFrame):
                        continue

                    norm_col = norm_by_session.get(sk)
                    mm_col = mm_by_session.get(sk)

                    if show_engineering:
                        if mm_col:
                            h = _extract_series(df, mm_col, include_inactive=False)
                            if h.size:
                                hist_chunks.append(h)
                    else:
                        if norm_col:
                            h = _extract_series(df, norm_col, include_inactive=False)
                            if h.size:
                                hist_chunks.append(h)

                    if norm_col and mm_col:
                        n_vals, m_vals = _extract_paired_series(
                            df,
                            norm_col=norm_col,
                            mm_col=mm_col,
                            include_inactive=False,
                        )
                        if n_vals.size:
                            norm_chunks.append(n_vals)
                            mm_chunks.append(m_vals)

                hist_values_by_entity[label] = np.concatenate(hist_chunks) if hist_chunks else np.array([], dtype=float)
                paired_norm_by_entity[label] = np.concatenate(norm_chunks) if norm_chunks else np.array([], dtype=float)
                paired_mm_by_entity[label] = np.concatenate(mm_chunks) if mm_chunks else np.array([], dtype=float)

            fig, ax = plt.subplots(figsize=(8.0, 4.2))
            plotted = 0
            local_y_max = 0.0

            if show_engineering:
                all_vals = [v for v in hist_values_by_entity.values() if v.size]
                if all_vals:
                    merged = np.concatenate(all_vals)
                    merged = merged[np.isfinite(merged)]
                    lo = float(np.min(merged)) if merged.size else 0.0
                    hi = float(np.max(merged)) if merged.size else 1.0
                    lo = min(0.0, lo)
                    if hi <= lo:
                        hi = lo + 1.0
                    hist_range = (lo, hi)
                else:
                    hist_range = (0.0, 1.0)
            else:
                hist_range = (0.0, 1.0)

            for label, vals in hist_values_by_entity.items():
                clean = np.asarray(vals, dtype=float)
                clean = clean[np.isfinite(clean)]
                if clean.size == 0:
                    continue

                hist, edges = np.histogram(clean, bins=int(bins), range=hist_range)
                props = hist.astype(float) / float(max(clean.size, 1))
                ax.stairs(props, edges, label=label, linewidth=1.4)
                if props.size:
                    local_y_max = max(local_y_max, float(np.max(props)))
                plotted += 1

            if y_key:
                shared[str(y_key)] = max(local_y_max, 0.0)

            target_y = local_y_max
            if shared:
                target_y = max([target_y] + [float(v) for v in shared.values()])

            if show_engineering:
                ax.set_title('Engineering displacement distribution')
                ax.set_xlabel('Displacement (mm)')
            else:
                ax.set_title('Normalized displacement distribution')
                ax.set_xlabel('Normalized displacement')
                ax.set_xlim(0.0, 1.0)

            ax.set_ylabel('Proportion')
            if target_y > 0:
                ax.set_ylim(0.0, target_y * 1.05)
            ax.grid(True, alpha=0.3)

            if plotted > 1:
                ax.legend(fontsize=9)

            if plotted == 0:
                ax.text(0.5, 0.5, 'No numeric values after filtering', ha='center', va='center', transform=ax.transAxes)
                ax.set_axis_off()

            plt.show()

            metric_lines = [f"<div><b>Metrics (trim cutoff = {trim_cutoff:.2f})</b></div>"]
            stats_by_label: dict[str, dict[str, float | bool]] = {}

            for label in sorted(set(paired_norm_by_entity.keys()) | set(paired_mm_by_entity.keys())):
                n_vals = paired_norm_by_entity.get(label, np.array([], dtype=float))
                m_vals = paired_mm_by_entity.get(label, np.array([], dtype=float))
                m = _paired_disp_metrics(n_vals, m_vals, trim_cutoff=float(trim_cutoff))
                stats_by_label[label] = m
                if bool(m['insufficient']):
                    metric_lines.append(f"<div style='margin-top:8px'><b>{label}</b><br>insufficient paired norm/mm data</div>")
                    continue

                metric_lines.append(
                    "<div style='margin-top:8px'>"
                    f"<b>{label}</b><br>"
                    f"Dynamic sag (Q50): {_fmt_pct_mm(float(m['q50_n']), float(m['q50_mm']))}<br>"
                    f"95th percentile (Q95): {_fmt_pct_mm(float(m['q95_n']), float(m['q95_mm']))}<br>"
                    f"Maximum travel (Q100): {_fmt_pct_mm(float(m['q100_n']), float(m['q100_mm']))}<br>"
                    f"Interquartile range (IQR): {_fmt_pct_mm(float(m['iqr_n']), float(m['iqr_mm']))}<br>"
                    f"Skew (skew): {_fmt_pct_mm(float(m['skew_n']), float(m['skew_mm']))}"
                    "</div>"
                )

            display(W.HTML(''.join(metric_lines)))
            state['stats'] = stats_by_label
            state['norm_by_session'] = dict(norm_by_session)
            state['mm_by_session'] = dict(mm_by_session)

    rebuild()
    return {'out': out, 'rebuild': rebuild, 'state': state}


In [3]:
w_row1_show_engineering = W.Checkbox(
    value=False,
    description='Show engineering units (mm)',
    indent=False,
)
display(w_row1_show_engineering)


row1_shared_y = {}


def _row1_show_engineering() -> bool:
    return bool(w_row1_show_engineering.value)


front_tile = make_displacement_hist_tile_rebuilder(
    sel=sel,
    side='front',
    title='Front Suspension: Displacement Histogram',
    y_shared=row1_shared_y,
    y_key='front',
    show_engineering_getter=_row1_show_engineering,
)

rear_tile = make_displacement_hist_tile_rebuilder(
    sel=sel,
    side='rear',
    title='Rear Suspension: Displacement Histogram',
    y_shared=row1_shared_y,
    y_key='rear',
    show_engineering_getter=_row1_show_engineering,
)

front_tile['out'].layout = W.Layout(width='49%')
rear_tile['out'].layout = W.Layout(width='49%')

first_row = W.HBox(
    [front_tile['out'], rear_tile['out']],
    layout=W.Layout(width='100%', align_items='flex-start', justify_content='space-between'),
)
display(first_row)


def _rebuild_row1(*_):
    row1_shared_y.clear()
    front_tile['rebuild']()
    rear_tile['rebuild']()
    front_tile['rebuild']()


_rebuild_row1()

w_row1_show_engineering.observe(_rebuild_row1, names='value')

if 'row1_refresh' in globals():
    try:
        row1_refresh['detach']()
    except Exception:
        pass

row1_refresh = attach_refresh(sel, rebuild_fns=[_rebuild_row1])


Checkbox(value=False, description='Show engineering units (mm)', indent=False)

## Velocity Histograms + Rebound/Compression Metrics

In [4]:
#Velocity signal selection is registry-driven (`domain=suspension`, `unit=mm/s`, side match).

#Fixed settings for both tiles:

#- Bins: 100
#- CDF: False
#- Normalize: True
#- Drop NaN/Inf: True
#- Include inactive: False
#- Show metrics: False (custom metrics shown below each chart)
#- Trim cutoff: 0
#- Stepped histogram with an overflow bin for values `>= 2000` mm/s


import numpy as np
import matplotlib.pyplot as plt


def _velocity_candidates_for_side(signals: Mapping[str, Mapping[str, Any]], side: str) -> list[str]:
    side_l = str(side).strip().lower()
    candidates: list[tuple[tuple[int, int, int, int], str]] = []

    for col, info in signals.items():
        if not isinstance(info, Mapping):
            continue
        if str(info.get('kind') or '').strip().lower() == 'qc':
            continue

        domain = str(info.get('domain') or '').strip().lower()
        if domain != 'suspension':
            continue

        col_s = str(col)
        col_l = col_s.lower()
        sensor = str(info.get('sensor') or '').strip().lower()
        if side_l not in col_l and side_l not in sensor:
            continue

        quantity = str(info.get('quantity') or '').strip().lower()
        unit = str(info.get('unit') or '').strip().lower()
        if unit != 'mm/s' and quantity != 'vel':
            continue

        score = (
            1 if quantity == 'vel' else 0,
            1 if unit == 'mm/s' else 0,
            1 if sensor.startswith(f'{side_l}_') else 0,
            1 if '_vel_' in col_l or col_l.endswith('_vel') else 0,
        )
        candidates.append((score, col_s))

    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return [col for _score, col in candidates]


def _resolve_velocity_source_by_session(*, session_loader, session_keys: list[str], side: str) -> tuple[dict[str, str], list[str]]:
    source_by_session: dict[str, str] = {}
    missing: list[str] = []
    for sk in map(str, session_keys):
        sess = session_loader(sk)
        meta = (sess or {}).get('meta') or {}
        signals = meta.get('signals') or {}
        if not isinstance(signals, Mapping):
            signals = {}
        cands = _velocity_candidates_for_side(signals, side)
        if not cands:
            missing.append(sk)
            continue
        source_by_session[sk] = cands[0]
    return source_by_session, missing


def _fmt(v: float) -> str:
    if np.isnan(v):
        return 'nan'
    return f'{v:.1f}'


def _phase_stats(vals: np.ndarray) -> dict[str, float]:
    a = np.asarray(vals, dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return {'n': 0.0, 'mean': np.nan, 'max_abs': np.nan, 'p95_abs': np.nan}
    abs_a = np.abs(a)
    return {
        'n': float(a.size),
        'mean': float(np.mean(a)),
        'max_abs': float(np.max(abs_a)),
        'p95_abs': float(np.quantile(abs_a, 0.95)),
    }


def _velocity_hist_proportions(vals: np.ndarray, *, bins: int, x_abs_limit: float) -> tuple[np.ndarray, np.ndarray, float]:
    clean = np.asarray(vals, dtype=float)
    clean = clean[np.isfinite(clean)]

    core_edges = np.linspace(-float(x_abs_limit), float(x_abs_limit), int(bins) + 1)
    core_width = float(core_edges[1] - core_edges[0]) if core_edges.size > 1 else max(1.0, float(x_abs_limit) / 25.0)
    overflow_width = max(core_width, float(x_abs_limit) * 0.10)

    right_edge = float(x_abs_limit) + overflow_width
    edges = np.concatenate((core_edges, [right_edge]))

    if clean.size == 0:
        return np.zeros(edges.size - 1, dtype=float), edges, overflow_width

    clipped = np.clip(clean, float(core_edges[0]) + 1e-9, right_edge - 1e-9)
    counts, _ = np.histogram(clipped, bins=edges)
    total = int(counts.sum())
    props = counts.astype(float) / float(total if total > 0 else 1)
    return props, edges, overflow_width


def make_velocity_hist_tile_rebuilder(
    *,
    sel,
    side: str,
    title: str,
    bins: int = 100,
    x_abs_limit: float = 2000.0,
    y_shared: dict[str, float] | None = None,
    y_key: str | None = None,
):
    out = W.Output(layout=W.Layout(border='1px solid #d9d9d9', padding='8px', width='100%'))
    state: dict[str, Any] = {}
    side_l = str(side).strip().lower()
    shared = y_shared if isinstance(y_shared, dict) else {}

    def rebuild() -> None:
        snapshot = entity_snapshot_from_handle(sel)
        selected_entities = list(snapshot.selected_entities)

        with out:
            clear_output(wait=True)
            display(W.HTML(f"<h4 style='margin:0 0 8px 0;'>{title}</h4>"))

            if not selected_entities:
                print('No entities selected.')
                return

            key_to_ref = dict(snapshot.key_to_ref)
            base_loader = make_session_loader(store=sel['store'], key_to_ref=key_to_ref)
            session_keys = list(map(str, snapshot.expanded_session_keys))
            source_by_session, missing = _resolve_velocity_source_by_session(
                session_loader=base_loader,
                session_keys=session_keys,
                side=side_l,
            )

            if not source_by_session:
                print(
                    'No matching suspension velocity signal found '
                    f'for side={side_l!r} in the current selection.'
                )
                return

            if missing:
                sample = ', '.join(missing[:3])
                display(
                    W.HTML(
                        f'<small><b>Note:</b> {len(missing)} session(s) had no side={side_l} velocity signal '
                        f'(examples: {sample}).</small>'
                    )
                )

            entity_values: dict[str, np.ndarray] = {}
            for entity in selected_entities:
                entity_key = str(entity.entity_key)
                label = str(entity.label)
                members = snapshot.entity_to_effective_members.get(entity_key, [entity_key])
                chunks: list[np.ndarray] = []

                for sk in map(str, members):
                    src = source_by_session.get(sk)
                    if not src:
                        continue
                    sess = base_loader(sk)
                    df = (sess or {}).get('df')
                    if not isinstance(df, pd.DataFrame) or src not in df.columns:
                        continue
                    s = pd.to_numeric(df[src], errors='coerce')
                    if 'active_mask_qc' in df.columns:
                        s = s[df['active_mask_qc'].astype(bool)]
                    vals = s.to_numpy(dtype=float, copy=False)
                    vals = vals[np.isfinite(vals)]
                    if vals.size:
                        chunks.append(vals)

                entity_values[label] = np.concatenate(chunks) if chunks else np.array([], dtype=float)

            fig, ax = plt.subplots(figsize=(8.0, 4.2))
            plotted = 0
            stats_by_label: dict[str, dict[str, dict[str, float]]] = {}
            last_edges: np.ndarray | None = None
            local_y_max = 0.0

            for label, vals in entity_values.items():
                clean = np.asarray(vals, dtype=float)
                clean = clean[np.isfinite(clean)]
                if clean.size == 0:
                    stats_by_label[label] = {
                        'rebound': _phase_stats(np.array([], dtype=float)),
                        'compression': _phase_stats(np.array([], dtype=float)),
                    }
                    continue

                props, edges, _overflow_width = _velocity_hist_proportions(
                    clean,
                    bins=int(bins),
                    x_abs_limit=float(x_abs_limit),
                )
                ax.stairs(props, edges, label=label, linewidth=1.4)
                if props.size:
                    local_y_max = max(local_y_max, float(np.max(props)))
                last_edges = edges
                plotted += 1

                stats_by_label[label] = {
                    'rebound': _phase_stats(clean[clean < 0]),
                    'compression': _phase_stats(clean[clean > 0]),
                }

            if y_key:
                shared[str(y_key)] = max(local_y_max, 0.0)

            target_y = local_y_max
            if shared:
                target_y = max([target_y] + [float(v) for v in shared.values()])

            ax.set_title('Velocity distribution')
            ax.set_xlabel('Velocity (mm/s)')
            ax.set_ylabel('Proportion')
            if target_y > 0:
                ax.set_ylim(0.0, target_y * 1.05)
            ax.grid(True, alpha=0.3)

            if last_edges is not None:
                ax.set_xlim(float(last_edges[0]), float(last_edges[-1]))
                ax.set_xticks(
                    [
                        -float(x_abs_limit),
                        0.0,
                        float(x_abs_limit),
                        float(last_edges[-1]),
                    ]
                )
                ax.set_xticklabels(
                    [
                        f'{-int(x_abs_limit)}',
                        '0',
                        f'{int(x_abs_limit)}',
                        '',
                    ]
                )
                ax.axvline(-float(x_abs_limit), color='#999999', linestyle=':', linewidth=1.0, alpha=0.9)
                ax.axvline(0.0, color='#777777', linestyle='--', linewidth=1.0, alpha=0.9)
                ax.axvline(float(x_abs_limit), color='#999999', linestyle=':', linewidth=1.0, alpha=0.9)

            if plotted > 1:
                ax.legend(fontsize=9)
            if plotted == 0:
                ax.text(0.5, 0.5, 'No numeric values after filtering', ha='center', va='center', transform=ax.transAxes)
                ax.set_axis_off()
            plt.show()

            left_lines = ['<div style="margin-left:12px"><b>Rebound (v &lt; 0)</b></div>']
            right_lines = ['<div><b>Compression (v &gt; 0)</b></div>']

            for label, phases in stats_by_label.items():
                r = phases['rebound']
                c = phases['compression']

                if r['n'] <= 0:
                    left_lines.append(f"<div style='margin-top:8px;margin-left:12px'><b>{label}</b><br>no rebound samples</div>")
                else:
                    left_lines.append(
                        "<div style='margin-top:8px;margin-left:12px'>"
                        f"<b>{label}</b><br>"
                        f"mean: {_fmt(r['mean'])} mm/s<br>"
                        f"max |v|: {_fmt(r['max_abs'])} mm/s<br>"
                        f"p95 |v|: {_fmt(r['p95_abs'])} mm/s"
                        "</div>"
                    )

                if c['n'] <= 0:
                    right_lines.append(f"<div style='margin-top:8px'><b>{label}</b><br>no compression samples</div>")
                else:
                    right_lines.append(
                        "<div style='margin-top:8px'>"
                        f"<b>{label}</b><br>"
                        f"mean: {_fmt(c['mean'])} mm/s<br>"
                        f"max |v|: {_fmt(c['max_abs'])} mm/s<br>"
                        f"p95 |v|: {_fmt(c['p95_abs'])} mm/s"
                        "</div>"
                    )

            metrics_row = W.HBox(
                [
                    W.HTML(''.join(left_lines), layout=W.Layout(width='50%')),
                    W.HTML(''.join(right_lines), layout=W.Layout(width='50%')),
                ],
                layout=W.Layout(width='100%', align_items='flex-start', justify_content='space-between'),
            )
            display(metrics_row)

            state['stats'] = stats_by_label
            state['source_by_session'] = dict(source_by_session)

    rebuild()
    return {'out': out, 'rebuild': rebuild, 'state': state}


row2_shared_y = {}

front_vel_tile = make_velocity_hist_tile_rebuilder(
    sel=sel,
    side='front',
    title='Front Suspension: Velocity',
    y_shared=row2_shared_y,
    y_key='front',
)

rear_vel_tile = make_velocity_hist_tile_rebuilder(
    sel=sel,
    side='rear',
    title='Rear Suspension: Velocity',
    y_shared=row2_shared_y,
    y_key='rear',
)

front_vel_tile['out'].layout = W.Layout(width='49%')
rear_vel_tile['out'].layout = W.Layout(width='49%')

second_row = W.HBox(
    [front_vel_tile['out'], rear_vel_tile['out']],
    layout=W.Layout(width='100%', align_items='flex-start', justify_content='space-between'),
)
display(second_row)


def _rebuild_row2():
    front_vel_tile['rebuild']()
    rear_vel_tile['rebuild']()
    front_vel_tile['rebuild']()


_rebuild_row2()

if 'row2_refresh' in globals():
    try:
        row2_refresh['detach']()
    except Exception:
        pass

row2_refresh = attach_refresh(sel, rebuild_fns=[_rebuild_row2])
